# Regresión Lineal Múltiple — Ejercicios prácticos

Objetivo: aprender a ajustar, diagnosticar e interpretar modelos de regresión lineal múltiple en Python, poniendo especial foco en entender los resultados de los summaries de statsmodels.

## 1) Preparación del entorno

In [ ]:
# Imports y configuración
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, normal_ad
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

np.random.seed(42)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set(style="whitegrid", context="notebook")

print("Entorno listo.")

## 2) Datos sintéticos de ejemplo (viviendas)
Generamos un dataset realista para modelar el precio de viviendas (en miles de euros) a partir de superficie, antigüedad, distancia al centro y zona (categoría). También creamos 'habitaciones' correlacionada con superficie para ilustrar multicolinealidad.

In [ ]:
# Generación de datos sintéticos
n = 220
superficie = np.clip(np.random.normal(90, 20, size=n), 45, 200)
antiguedad = np.clip(np.random.normal(12, 7, size=n), 0, 40)
distancia = np.clip(np.random.normal(3.0, 1.2, size=n), 0.2, 10)
zona = np.random.choice(
    ["periferia", "semicentro", "centro"], size=n, p=[0.45, 0.35, 0.20]
)

# Habitaciones correlacionadas con superficie
habitaciones = np.round(superficie / 20 + np.random.normal(0, 0.6, size=n)).astype(int)
habitaciones = np.clip(habitaciones, 1, 8)

# Efectos reales (desconocidos en la práctica)
beta0 = 50  # intercepto (miles €)
beta_superficie = 2.8
beta_antiguedad = -1.1
beta_distancia = -7.5
efecto_zona = {"periferia": 0, "semicentro": 15, "centro": 35}

# Precio con ruido
epsilon = np.random.normal(0, 15, size=n)
precio = (
    beta0
    + beta_superficie * superficie
    + beta_antiguedad * antiguedad
    + beta_distancia * distancia
    + np.array([efecto_zona[z] for z in zona])
    + epsilon
)

df = pd.DataFrame(
    {
        "precio": precio,
        "superficie": superficie,
        "antiguedad": antiguedad,
        "distancia": distancia,
        "zona": zona,
        "habitaciones": habitaciones,
    }
)

df.head()

### Exploración rápida

In [ ]:
display(df.describe(include="all"))

plt.figure(figsize=(12, 4))
sns.heatmap(
    df[["precio", "superficie", "antiguedad", "distancia", "habitaciones"]].corr(),
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)
plt.title("Matriz de correlaciones")
plt.show()

## 3) Modelo base: precio ~ superficie + antigüedad + distancia
Ajustamos un OLS y analizamos el summary.

In [ ]:
modelo_base = smf.ols("precio ~ superficie + antiguedad + distancia", data=df).fit()
print(modelo_base.summary())

### ¿Cómo leer el summary?
- R-squared (R²): proporción de variabilidad explicada por el modelo.
- Adj. R-squared: R² penalizado por nº de predictores; mejor para comparar modelos con diferente complejidad.
- F-statistic y Prob (F-statistic): test global de que al menos un coeficiente es distinto de 0.
- coef: valor estimado de cada parámetro (intercepto y pendientes).
- std err: error estándar del coeficiente (incertidumbre de la estimación).
- t y P>|t|: significación individual de cada coeficiente (hipótesis H0: beta=0).
- [0.025, 0.975]: intervalo de confianza al 95% para el coeficiente.
- AIC/BIC: criterios de información (menor = mejor, a igual capacidad predictiva).
- Omnibus, Jarque-Bera, Skew, Kurtosis: diagnóstico de normalidad de residuos.
- Cond. No.: indica posible multicolinealidad si es muy alto.

In [ ]:
# Extraer y resumir métricas clave programáticamente
res = modelo_base
metrics = {
    "R2": res.rsquared,
    "Adj_R2": res.rsquared_adj,
    "F_stat": res.fvalue,
    "F_pvalue": res.f_pvalue,
    "AIC": res.aic,
    "BIC": res.bic,
}
coef_table = pd.concat(
    [res.params, res.bse, res.tvalues, res.pvalues, res.conf_int()], axis=1
)
coef_table.columns = ["coef", "std_err", "t", "p", "ci_lower", "ci_upper"]

display(pd.Series(metrics).to_frame("valor"))
display(coef_table)

## 4) Añadir variable categórica (zona) con dummies
Usamos notación de fórmula con `C(zona)` para crear dummies automáticamente. La categoría de referencia se elige alfabéticamente por defecto (a veces conviene fijarla manualmente).

In [ ]:
modelo_zona = smf.ols('precio ~ superficie + antiguedad + distancia + C(zona)', data=df).fit()
print(modelo_zona.summary())

# Interpretación programática de los efectos de zona
params = modelo_zona.params
base = 'C(zona)[T.centro]' if 'C(zona)[T.centro]' in params else (
        'C(zona)[T.periferia]' if 'C(zona)[T.periferia]' in params else 'C(zona)[T.semicentro]')
print('
Coeficientes de zona (respecto a la referencia):')
for k in params.index:
    if k.startswith('C(zona)'):
        print(f'  {k}: {params[k]:.3f} (p={modelo_zona.pvalues[k]:.3f})')
print(f

## 5) Multicolinealidad: añadir 'habitaciones' y calcular VIF
Al incluir 'habitaciones' (muy correlacionada con 'superficie'), los errores estándar pueden inflarse y la interpretación puede volverse inestable. Veamos los VIF.

In [ ]:
# Modelo con multicolinealidad potencial
modelo_multi = smf.ols(
    "precio ~ superficie + antiguedad + distancia + habitaciones", data=df
).fit()
print(modelo_multi.summary())

# VIF: se calcula sobre matriz de predictores con constante
X = sm.add_constant(df[["superficie", "antiguedad", "distancia", "habitaciones"]])
vifs = pd.Series(
    [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    index=X.columns,
    name="VIF",
)
vifs

Interpretación rápida del VIF:
- VIF ≈ 1–5: aceptable.
- VIF > 10: multicolinealidad preocupante.

Si hay VIF altos, considera eliminar/combinar variables o aplicar regularización.

## 6) Interacciones: efecto de superficie según zona
Incluimos una interacción `superficie:C(zona)` para permitir que la pendiente de superficie cambie según la zona.

In [ ]:
modelo_inter = smf.ols(
    "precio ~ superficie * C(zona) + antiguedad + distancia", data=df
).fit()
print(modelo_inter.summary())

# Nota interpretativa: el coeficiente de 'superficie' corresponde a la pendiente en la categoría de referencia de zona.
# Los términos de interacción ajustan esa pendiente en las otras categorías.

## 7) Diagnóstico del modelo (residuos)
Comprobamos supuestos con gráficos y tests: normalidad (Anderson-Darling) y heterocedasticidad (Breusch-Pagan).

In [ ]:
res = modelo_zona  # Puedes cambiar aquí el modelo a diagnosticar
residuos = res.resid
ajustados = res.fittedvalues

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].scatter(ajustados, residuos, alpha=0.7)
axes[0].axhline(0, color='red', ls='--')
axes[0].set_xlabel('Valores ajustados')
axes[0].set_ylabel('Residuos')
axes[0].set_title('Residuos vs Ajustados')

sm.qqplot(residuos, line='45', ax=axes[1])
axes[1].set_title('Q-Q plot de residuos')
plt.tight_layout(); plt.show()

# Test de normalidad (Anderson-Darling aproximado)
stat_ad, p_ad = normal_ad(residuos)
print(f'Normalidad (AD): estadístico={stat_ad:.3f}, p-valor={p_ad:.3f}')

# Test de heterocedasticidad (Breusch-Pagan)
bp_stat, bp_p, _, _ = het_breuschpagan(residuos, res.model.exog)
print(f'Heterocedasticidad (BP): estadístico={bp_stat:.3f}, p-valor={bp_p:.3f}')

print('
Criterio práctico:')
print('- p>0.05 en normalidad: no evidencia fuerte contra normalidad de residuos.')
print('- p>0.05 en BP: no evidencia fuerte de heterocedasticidad.')

## 8) Evaluación predictiva con train/test
Dividimos datos y medimos el error (RMSE) para evaluar capacidad de generalización.

In [ ]:
train, test = train_test_split(df, test_size=0.25, random_state=42)

fml = 'precio ~ superficie + antiguedad + distancia + C(zona)'
m_train = smf.ols(fml, data=train).fit()
pred = m_train.predict(test)
rmse = mean_squared_error(test['precio'], pred, squared=False)
print(f'RMSE test: {rmse:.3f} (miles de €)')

# Comparar con modelo sin zona
m_train_base = smf.ols('precio ~ superficie + antiguedad + distancia', data=train).fit()
pred_base = m_train_base.predict(test)
rmse_base = mean_squared_error(test['precio'], pred_base, squared=False)
print(f'RMSE test (sin zona): {rmse_base:.3f} (miles de €)')

print('
¿Añadir zona mejora el RMSE en test? Comparar valores.')

## 9) Comparación de modelos (AIC, BIC, Adj R²)
Creamos una tabla para comparar varios modelos. Menor AIC/BIC y mayor Adj R² suelen ser preferibles, con prudencia para evitar sobreajuste.

In [ ]:
comparacion = pd.DataFrame(
    {
        "modelo": ["base", "con_zona", "multi_corr", "interaccion"],
        "AIC": [modelo_base.aic, modelo_zona.aic, modelo_multi.aic, modelo_inter.aic],
        "BIC": [modelo_base.bic, modelo_zona.bic, modelo_multi.bic, modelo_inter.bic],
        "Adj_R2": [
            modelo_base.rsquared_adj,
            modelo_zona.rsquared_adj,
            modelo_multi.rsquared_adj,
            modelo_inter.rsquared_adj,
        ],
    }
).sort_values("AIC")
comparacion

## 10) Guía rápida para interpretar summaries (cheatsheet)
- ¿Es significativo el modelo completo? F-statistic y su p-valor.
- ¿Qué tan bien explica? R² y R² ajustado (ojo: comparar entre modelos).
- ¿Qué predictores son significativos? t y P>|t| por variable.
- ¿Signos coherentes? Verifica que las direcciones (coef) tengan sentido de negocio.
- ¿Intervalos razonables? CI al 95% que no crucen 0 para efectos claros.
- ¿Supuestos razonables? Revisa residuos, normalidad y homocedasticidad.
- ¿Multicolinealidad? VIF y número de condición.
- ¿Parcimonia? Compara AIC/BIC y evita añadir variables irrelevantes.

## 11) Ejercicios propuestos (tu turno)
1. Quita 'distancia' del modelo con zona. ¿Cómo cambian R² ajustado y AIC/BIC?
2. Prueba una transformación logarítmica: `precio_log = log(precio)` y ajusta un modelo. Compara diagnóstico de residuos.
3. Añade 'habitaciones' junto con 'superficie'. Calcula VIF y comenta el efecto en coeficientes y p-valores.
4. Incluye una interacción `antiguedad:C(zona)`. ¿Hay evidencia de que el efecto de la antigüedad dependa de la zona?
5. Divide en train/test con otro `random_state` y compara RMSE con y sin interacción.
6. Cambia la referencia de `zona` (por ejemplo, usa 'centro' como referencia) y reinterpreta los coeficientes.

Consejo: documenta brevemente tu interpretación bajo cada celda.

In [ ]:
# Ejercicio 1: elimina 'distancia' y compara
modelo_sin_dist = smf.ols("precio ~ superficie + antiguedad + C(zona)", data=df).fit()
display(
    pd.Series(
        {
            "base_Adj_R2": modelo_zona.rsquared_adj,
            "nuevo_Adj_R2": modelo_sin_dist.rsquared_adj,
            "base_AIC": modelo_zona.aic,
            "nuevo_AIC": modelo_sin_dist.aic,
            "base_BIC": modelo_zona.bic,
            "nuevo_BIC": modelo_sin_dist.bic,
        }
    ).to_frame("valor")
)
print(modelo_sin_dist.summary())  # Explica los cambios observados